In [ ]:
import sys
import os

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nCurrent working directory:")
print(os.getcwd())

In [ ]:
from pathlib import Path

# Find the project directory
current_dir = Path.cwd()

project_root = None
dataset_dir = None

for folder in [current_dir, *current_dir.parents]:
    candidate = folder / "data" / "raw" / "Dataset_BUSI_with_GT"

    if candidate.is_dir():
        project_root = folder
        dataset_dir = candidate
        break

# Verify dataset availability
if dataset_dir is None:
    print("Dataset directory not found!")
    print("Current working directory:", current_dir)

else:
    print("Dataset found successfully!")
    print("Project root:", project_root)
    print("Dataset location:", dataset_dir)

    print("\nDataset folders:")

    for folder in sorted(dataset_dir.iterdir()):
        if folder.is_dir():
            print(" -", folder.name)

In [ ]:
# Define dataset categories
classes = ["benign", "malignant", "normal"]

# Supported image formats
image_extensions = {".png", ".jpg", ".jpeg", ".bmp"}

# Store dataset statistics
dataset_stats = {}

for class_name in classes:

    class_dir = dataset_dir / class_name

    # Get all image files
    files = [
        file for file in class_dir.iterdir()
        if file.is_file() and file.suffix.lower() in image_extensions
    ]

    # Separate ultrasound images and segmentation masks
    masks = [file for file in files if "_mask" in file.stem]
    images = [file for file in files if "_mask" not in file.stem]

    dataset_stats[class_name] = {
        "images": len(images),
        "masks": len(masks)
    }

# Display results
print("BUSI DATASET STATISTICS\n")

for class_name, stats in dataset_stats.items():
    print(f"{class_name.upper()}:")
    print(f"  Ultrasound images: {stats['images']}")
    print(f"  Segmentation masks: {stats['masks']}")
    print()

total_images = sum(x["images"] for x in dataset_stats.values())
total_masks = sum(x["masks"] for x in dataset_stats.values())

print("--------------------------------")
print("TOTAL ULTRASOUND IMAGES:", total_images)
print("TOTAL SEGMENTATION MASKS:", total_masks)

In [ ]:
# Inspect filenames in each category

for class_name in classes:

    class_dir = dataset_dir / class_name

    print(f"\n{'='*50}")
    print(f"{class_name.upper()} - FIRST 15 FILES")
    print('='*50)

    files = sorted(class_dir.iterdir())

    for file in files[:15]:
        if file.is_file():
            print(file.name)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Select an example image
class_name = "benign"
image_name = "benign (100).png"

# Define paths
class_dir = dataset_dir / class_name

image_path = class_dir / image_name

# Find all masks belonging to this image
mask_paths = sorted(
    class_dir.glob("benign (100)_mask*.png")
)

# Load original ultrasound image
original_image = np.array(Image.open(image_path))

# Load segmentation masks
masks = [
    np.array(Image.open(mask_path))
    for mask_path in mask_paths
]

# Display image and masks
fig, axes = plt.subplots(
    1, len(masks) + 1,
    figsize=(15, 5)
)

axes[0].imshow(original_image, cmap="gray")
axes[0].set_title("Original Ultrasound Image")
axes[0].axis("off")

for i, mask in enumerate(masks):

    axes[i + 1].imshow(mask, cmap="gray")
    axes[i + 1].set_title(f"Mask {i + 1}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

print("Original image shape:", original_image.shape)

for i, mask in enumerate(masks):
    print(f"Mask {i + 1} shape:", mask.shape)

In [ ]:
# Inspect pixel properties of each mask

for i, mask in enumerate(masks):

    print(f"\nMASK {i + 1}")

    print("Data type:", mask.dtype)
    print("Minimum pixel value:", mask.min())
    print("Maximum pixel value:", mask.max())

    print("Unique pixel values:", np.unique(mask))

    # Count background and foreground pixels
    background_pixels = np.sum(mask == 0)
    foreground_pixels = np.sum(mask > 0)

    print("Background pixels:", background_pixels)
    print("Foreground pixels:", foreground_pixels)

In [ ]:
# Combine all masks using logical OR

combined_mask = np.logical_or.reduce(masks)

# Inspect the combined mask
print("Combined mask properties")
print("------------------------")

print("Data type:", combined_mask.dtype)
print("Shape:", combined_mask.shape)

print("Unique pixel values:", np.unique(combined_mask))

print("Total foreground pixels:", np.sum(combined_mask))
print("Total background pixels:", np.sum(~combined_mask))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_image)
axes[0].set_title("Original Ultrasound")

axes[1].imshow(combined_mask, cmap="gray")
axes[1].set_title("Combined Mask")

axes[2].imshow(original_image)
axes[2].imshow(combined_mask, cmap="Reds", alpha=0.4)
axes[2].set_title("Lesion Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Store image-mask relationships
image_mask_pairs = {}

# Store audit statistics
audit_stats = {}

for class_name in classes:

    class_dir = dataset_dir / class_name

    images = sorted([
        f for f in class_dir.iterdir()
        if f.is_file()
        and f.suffix.lower() in image_extensions
        and "_mask" not in f.stem
    ])

    no_mask = 0
    single_mask = 0
    multiple_masks = 0
    dimension_mismatch = 0

    for image_path in images:

        # Find masks belonging to this image
        mask_paths = sorted(
            class_dir.glob(f"{image_path.stem}_mask*.png")
        )

        # Store image-mask relationship
        image_mask_pairs[str(image_path)] = mask_paths

        if len(mask_paths) == 0:
            no_mask += 1

        elif len(mask_paths) == 1:
            single_mask += 1

        else:
            multiple_masks += 1

        # Check image and mask dimensions
        with Image.open(image_path) as img:
            image_size = img.size

        for mask_path in mask_paths:

            with Image.open(mask_path) as mask:
                mask_size = mask.size

            if image_size != mask_size:
                dimension_mismatch += 1

    audit_stats[class_name] = {
        "images": len(images),
        "no_mask": no_mask,
        "single_mask": single_mask,
        "multiple_masks": multiple_masks,
        "dimension_mismatch": dimension_mismatch
    }

# Display audit results
for class_name, stats in audit_stats.items():

    print(f"\n{class_name.upper()}")
    print("-" * 35)

    for key, value in stats.items():
        print(f"{key}: {value}")

print("\nTOTAL IMAGE-MASK PAIRS:", len(image_mask_pairs))

In [ ]:
# Find images having more than one mask

multiple_mask_images = {
    image_path: mask_paths
    for image_path, mask_paths in image_mask_pairs.items()
    if len(mask_paths) > 1
}

print("Images with multiple masks:", len(multiple_mask_images))

print("\nMULTIPLE MASK DETAILS")
print("-" * 50)

for image_path, mask_paths in multiple_mask_images.items():

    print(f"\nImage: {Path(image_path).name}")
    print("Number of masks:", len(mask_paths))

    for mask_path in mask_paths:
        print("  -", mask_path.name)


# Calculate total masks found by our matching function
total_matched_masks = sum(
    len(mask_paths)
    for mask_paths in image_mask_pairs.values()
)

print("\nTOTAL MATCHED MASKS:", total_matched_masks)

In [ ]:
# Find images having more than two masks

for image_path, mask_paths in image_mask_pairs.items():

    if len(mask_paths) > 2:

        print("Image:", Path(image_path).name)
        print("Number of masks:", len(mask_paths))

        for mask_path in mask_paths:
            print("  ", mask_path.name)

In [ ]:
from collections import Counter

# Store image dimensions
image_dimensions = []

for image_path in image_mask_pairs.keys():

    with Image.open(image_path) as img:
        image_dimensions.append(img.size)

# Count occurrences of each dimension
dimension_counts = Counter(image_dimensions)

print("TOTAL IMAGES:", len(image_dimensions))

print("\nUNIQUE IMAGE DIMENSIONS:")
print("------------------------")

for dimension, count in dimension_counts.most_common():
    print(f"Width: {dimension[0]}, Height: {dimension[1]} --> {count} images")

print("\nNUMBER OF UNIQUE DIMENSIONS:", len(dimension_counts))

In [ ]:
# Convert image dimensions into NumPy array
dimensions_array = np.array(image_dimensions)

# Extract widths and heights
widths = dimensions_array[:, 0]
heights = dimensions_array[:, 1]

# Calculate aspect ratios
aspect_ratios = widths / heights

print("IMAGE DIMENSION ANALYSIS")
print("=" * 40)

print("\nWIDTH:")
print("Minimum:", widths.min())
print("Maximum:", widths.max())
print("Average:", round(widths.mean(), 2))

print("\nHEIGHT:")
print("Minimum:", heights.min())
print("Maximum:", heights.max())
print("Average:", round(heights.mean(), 2))

print("\nASPECT RATIO:")
print("Minimum:", round(aspect_ratios.min(), 3))
print("Maximum:", round(aspect_ratios.max(), 3))
print("Average:", round(aspect_ratios.mean(), 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Width distribution
axes[0].hist(widths, bins=30, edgecolor="black")
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Number of Images")

# Height distribution
axes[1].hist(heights, bins=30, edgecolor="black")
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height (pixels)")
axes[1].set_ylabel("Number of Images")

# Aspect ratio distribution
axes[2].hist(aspect_ratios, bins=30, edgecolor="black")
axes[2].set_title("Aspect Ratio Distribution")
axes[2].set_xlabel("Width / Height")
axes[2].set_ylabel("Number of Images")

plt.tight_layout()
plt.show()

In [ ]:
print("IMAGE DIMENSION PERCENTILES")
print("=" * 40)

for percentile in [25, 50, 75, 90, 95, 99]:

    width_value = np.percentile(widths, percentile)
    height_value = np.percentile(heights, percentile)

    print(
        f"{percentile}th percentile | "
        f"Width: {width_value:.1f} | "
        f"Height: {height_value:.1f}"
    )

In [ ]:
from pathlib import Path
from PIL import Image

dataset_dir = Path("data/raw/Dataset_BUSI_with_GT")

records = []

for category in ["benign", "malignant", "normal"]:

    category_dir = dataset_dir / category

    for image_path in sorted(category_dir.glob("*.png")):

        # Skip segmentation masks
        if "_mask" in image_path.stem.lower():
            continue

        with Image.open(image_path) as img:
            width, height = img.size

        records.append({
            "category": category,
            "filename": image_path.name,
            "width": width,
            "height": height,
            "aspect_ratio": width / height
        })


def print_extremes(metric, label):

    ordered = sorted(records, key=lambda x: x[metric])

    print(f"\n{label}")
    print("=" * 75)

    print("\n5 smallest:")

    for item in ordered[:5]:
        print(
            f"{item['category']:10s} | "
            f"{item['filename']:30s} | "
            f"{item['width']:4d} x {item['height']:4d} | "
            f"AR: {item['aspect_ratio']:.3f}"
        )

    print("\n5 largest:")

    for item in ordered[-5:]:
        print(
            f"{item['category']:10s} | "
            f"{item['filename']:30s} | "
            f"{item['width']:4d} x {item['height']:4d} | "
            f"AR: {item['aspect_ratio']:.3f}"
        )


print("TOTAL IMAGES:", len(records))

print_extremes("width", "WIDTH EXTREMES")
print_extremes("height", "HEIGHT EXTREMES")
print_extremes("aspect_ratio", "ASPECT RATIO EXTREMES")

In [ ]:
from pathlib import Path

dataset_dir = Path("data/raw/Dataset_BUSI_with_GT")

print("CURRENT WORKING DIRECTORY:")
print(Path.cwd())

print("\nDATASET PATH:")
print(dataset_dir.resolve())

print("\nDOES DATASET DIRECTORY EXIST?")
print(dataset_dir.exists())

for category in ["benign", "malignant", "normal"]:

    category_dir = dataset_dir / category

    print(f"\n--- {category.upper()} ---")
    print("Directory exists:", category_dir.exists())

    if category_dir.exists():

        all_files = list(category_dir.iterdir())

        print("Total files:", len(all_files))
        print("First 5 files:", [f.name for f in all_files[:5]])

        print(
            "File extensions:",
            {f.suffix for f in all_files if f.is_file()}
        )

In [ ]:
from pathlib import Path
from PIL import Image

# Move one level up from notebooks to the project root
project_dir = Path.cwd().parent

# Construct the correct dataset path
dataset_dir = project_dir / "data" / "raw" / "Dataset_BUSI_with_GT"

print("PROJECT DIRECTORY:", project_dir)
print("DATASET DIRECTORY:", dataset_dir.resolve())
print("DATASET EXISTS:", dataset_dir.exists())

# Stop if the directory cannot be found
if not dataset_dir.exists():
    raise FileNotFoundError("BUSI dataset directory not found!")

records = []

for category in ["benign", "malignant", "normal"]:

    category_dir = dataset_dir / category

    for image_path in sorted(category_dir.iterdir()):

        # Skip segmentation masks
        if "_mask" in image_path.stem.lower():
            continue

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
            continue

        with Image.open(image_path) as img:
            width, height = img.size

        records.append({
            "category": category,
            "filename": image_path.name,
            "width": width,
            "height": height,
            "aspect_ratio": width / height
        })


print("\nTOTAL IMAGES:", len(records))


def print_extremes(metric, label):

    ordered = sorted(records, key=lambda x: x[metric])

    print(f"\n{label}")
    print("=" * 75)

    print("\n5 smallest:")

    for item in ordered[:5]:
        print(
            f"{item['category']:10s} | "
            f"{item['filename']:30s} | "
            f"{item['width']:4d} x {item['height']:4d} | "
            f"AR: {item['aspect_ratio']:.3f}"
        )

    print("\n5 largest:")

    for item in ordered[-5:]:
        print(
            f"{item['category']:10s} | "
            f"{item['filename']:30s} | "
            f"{item['width']:4d} x {item['height']:4d} | "
            f"AR: {item['aspect_ratio']:.3f}"
        )


print_extremes("width", "WIDTH EXTREMES")
print_extremes("height", "HEIGHT EXTREMES")
print_extremes("aspect_ratio", "ASPECT RATIO EXTREMES")

In [ ]:
# Find images with extreme dimensions
extremes = [
    ("Smallest width", min(records, key=lambda x: x["width"])),
    ("Largest width", max(records, key=lambda x: x["width"])),
    ("Smallest height", min(records, key=lambda x: x["height"])),
    ("Largest height", max(records, key=lambda x: x["height"])),
    ("Smallest aspect ratio", min(records, key=lambda x: x["aspect_ratio"])),
    ("Largest aspect ratio", max(records, key=lambda x: x["aspect_ratio"]))
]

# Remove duplicate selections
selected = {}

for label, item in extremes:
    key = (item["category"], item["filename"])
    selected.setdefault(key, (label, item))

# Display images
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (label, item) in zip(axes.flat, selected.values()):

    image_path = dataset_dir / item["category"] / item["filename"]

    img = Image.open(image_path)

    ax.imshow(img)
    ax.set_title(
        f"{label}\n{item['filename']}\n"
        f"{item['width']} x {item['height']}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from PIL import Image

print("CLASS DISTRIBUTION AND MASK AUDIT")
print("=" * 65)

for category in ["benign", "malignant", "normal"]:

    category_dir = dataset_dir / category

    images = [
        p for p in category_dir.glob("*.png")
        if "_mask" not in p.stem.lower()
    ]

    missing_masks = 0
    empty_masks = 0
    nonempty_masks = 0

    for image_path in images:

        mask_paths = sorted(
            category_dir.glob(f"{image_path.stem}_mask*.png")
        )

        if not mask_paths:
            missing_masks += 1
            continue

        combined_mask = None

        for mask_path in mask_paths:

            mask = np.array(
                Image.open(mask_path).convert("L")
            ) > 0

            if combined_mask is None:
                combined_mask = mask.copy()
            else:
                combined_mask = np.logical_or(
                    combined_mask, mask
                )

        if np.any(combined_mask):
            nonempty_masks += 1
        else:
            empty_masks += 1

    print(f"\n{category.upper()}")
    print("Images:", len(images))
    print("Missing masks:", missing_masks)
    print("Empty combined masks:", empty_masks)
    print("Non-empty combined masks:", nonempty_masks)

In [ ]:
import hashlib
import numpy as np

# Store hashes
exact_hashes = {}
difference_hashes = {}

for item in records:

    image_path = dataset_dir / item["category"] / item["filename"]

    # Exact file hash
    with open(image_path, "rb") as f:
        exact_hash = hashlib.sha256(f.read()).hexdigest()

    exact_hashes.setdefault(exact_hash, []).append(item)

    # Perceptual difference hash
    img = Image.open(image_path).convert("L").resize((9, 8))
    pixels = np.array(img)

    dhash = pixels[:, 1:] > pixels[:, :-1]
    difference_hashes[item["filename"], item["category"]] = dhash.flatten()


# Find exact duplicates
exact_duplicates = [
    group for group in exact_hashes.values()
    if len(group) > 1
]

print("EXACT DUPLICATE GROUPS:", len(exact_duplicates))

for group in exact_duplicates:
    print([x["category"] + "/" + x["filename"] for x in group])


# Find near-duplicate candidates
items = list(difference_hashes.items())
near_duplicates = []

for i in range(len(items)):
    for j in range(i + 1, len(items)):

        hash1 = items[i][1]
        hash2 = items[j][1]

        distance = np.count_nonzero(hash1 != hash2)

        if distance <= 5:
            near_duplicates.append(
                (items[i][0], items[j][0], distance)
            )


print("\nNEAR-DUPLICATE CANDIDATES:", len(near_duplicates))

for first, second, distance in near_duplicates[:30]:
    print(first, "<->", second, "| Distance:", distance)

if len(near_duplicates) > 30:
    print("Showing first 30 candidates only.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

duplicate_pairs = [
    ("benign", "benign (433).png"),
    ("malignant", "malignant (145).png")
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for row, (category, filename) in enumerate(duplicate_pairs):

    image_path = dataset_dir / category / filename

    image = np.array(Image.open(image_path).convert("RGB"))

    mask_paths = sorted(
        (dataset_dir / category).glob(
            f"{Path(filename).stem}_mask*.png"
        )
    )

    combined_mask = np.zeros(image.shape[:2], dtype=bool)

    for mask_path in mask_paths:
        mask = np.array(
            Image.open(mask_path).convert("L")
        ) > 0

        combined_mask |= mask

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"{category}: {filename}")

    axes[row, 1].imshow(combined_mask, cmap="gray")
    axes[row, 1].set_title("Combined segmentation mask")

    print(category, "| Masks:", len(mask_paths),
          "| Foreground pixels:", combined_mask.sum())

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

mask_paths = [
    dataset_dir / "benign" / "benign (433)_mask.png",
    dataset_dir / "malignant" / "malignant (145)_mask.png"
]

mask_a = np.array(Image.open(mask_paths[0]).convert("L")) > 0
mask_b = np.array(Image.open(mask_paths[1]).convert("L")) > 0

intersection = np.logical_and(mask_a, mask_b).sum()
union = np.logical_or(mask_a, mask_b).sum()

dice = (2 * intersection) / (mask_a.sum() + mask_b.sum())
iou = intersection / union

print("MASK COMPARISON")
print("=" * 40)
print("Mask A pixels:", mask_a.sum())
print("Mask B pixels:", mask_b.sum())
print("Intersection:", intersection)
print("Union:", union)
print("Different pixels:", np.logical_xor(mask_a, mask_b).sum())
print(f"Dice agreement: {dice:.4f}")
print(f"IoU agreement: {iou:.4f}")

# Visualize differences
comparison = np.zeros((*mask_a.shape, 3), dtype=np.uint8)
comparison[..., 0] = mask_a * 255
comparison[..., 1] = mask_b * 255

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(mask_a, cmap="gray")
axes[0].set_title("Benign mask")

axes[1].imshow(mask_b, cmap="gray")
axes[1].set_title("Malignant mask")

axes[2].imshow(comparison)
axes[2].set_title("Mask differences")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Separate same-category and cross-category candidates

cross_category = [
    pair for pair in near_duplicates
    if pair[0][1] != pair[1][1]
]

same_category = [
    pair for pair in near_duplicates
    if pair[0][1] == pair[1][1]
]

print("SAME-CATEGORY CANDIDATES:", len(same_category))
print("CROSS-CATEGORY CANDIDATES:", len(cross_category))

print("\nCROSS-CATEGORY CANDIDATES")
print("=" * 65)

for first, second, distance in sorted(
    cross_category, key=lambda x: x[2]
)[:30]:

    print(
        f"{first[1]}/{first[0]} <-> "
        f"{second[1]}/{second[0]} | "
        f"Distance: {distance}"
    )

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from skimage.metrics import structural_similarity

# --------------------------------------------------
# 1. IMAGE COMPARISON FUNCTIONS
# --------------------------------------------------

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def normalize_image(path, size=256):

    with Image.open(path) as img:
        img = img.convert("L")
        img.thumbnail(
            (size, size),
            Image.Resampling.LANCZOS
        )

        canvas = Image.new("L", (size, size), 0)

        x = (size - img.width) // 2
        y = (size - img.height) // 2

        canvas.paste(img, (x, y))

    return np.array(canvas)


# --------------------------------------------------
# 2. AUDIT ALL NEAR-DUPLICATE CANDIDATES
# --------------------------------------------------

audit_results = []

for first, second, dhash_distance in near_duplicates:

    filename1, category1 = first
    filename2, category2 = second

    path1 = dataset_dir / category1 / filename1
    path2 = dataset_dir / category2 / filename2

    image1 = normalize_image(path1)
    image2 = normalize_image(path2)

    ssim = structural_similarity(
        image1,
        image2,
        data_range=255
    )

    file_identical = (
        get_file_hash(path1) == get_file_hash(path2)
    )

    with Image.open(path1) as im1, Image.open(path2) as im2:
        original1 = np.array(im1.convert("RGB"))
        original2 = np.array(im2.convert("RGB"))

    pixel_identical = (
        original1.shape == original2.shape
        and np.array_equal(original1, original2)
    )

    audit_results.append({

        "category_1": category1,
        "filename_1": filename1,

        "category_2": category2,
        "filename_2": filename2,

        "dhash_distance": dhash_distance,

        "ssim": round(ssim, 5),

        "file_identical": file_identical,
        "pixel_identical": pixel_identical,

        "cross_category": category1 != category2

    })


# --------------------------------------------------
# 3. CREATE RESULTS TABLE
# --------------------------------------------------

audit_df = pd.DataFrame(audit_results)

audit_df = audit_df.sort_values(
    by="ssim",
    ascending=False
).reset_index(drop=True)

print("DUPLICATE AUDIT SUMMARY")
print("=" * 60)

print("Total candidates:", len(audit_df))

print(
    "Exact file duplicates:",
    audit_df["file_identical"].sum()
)

print(
    "Pixel-identical images:",
    audit_df["pixel_identical"].sum()
)

print(
    "High similarity (SSIM >= 0.98):",
    (audit_df["ssim"] >= 0.98).sum()
)

print(
    "Similarity requiring review (SSIM >= 0.90):",
    (audit_df["ssim"] >= 0.90).sum()
)


# --------------------------------------------------
# 4. CROSS-CATEGORY REPORT
# --------------------------------------------------

print("\nCROSS-CATEGORY DUPLICATE CANDIDATES")
print("=" * 60)

cross_df = audit_df[
    audit_df["cross_category"]
]

print(
    cross_df[
        [
            "filename_1",
            "category_1",
            "filename_2",
            "category_2",
            "ssim",
            "file_identical"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------
# 5. SAVE COMPLETE AUDIT REPORT
# --------------------------------------------------

output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

report_path = output_dir / "duplicate_audit.csv"

audit_df.to_csv(report_path, index=False)

print("\nREPORT SAVED:")
print(report_path)


# --------------------------------------------------
# 6. VISUALIZE ALL CROSS-CATEGORY CANDIDATES
# --------------------------------------------------

if len(cross_df) > 0:

    fig, axes = plt.subplots(
        len(cross_df),
        2,
        figsize=(12, 3 * len(cross_df))
    )

    if len(cross_df) == 1:
        axes = np.array([axes])

    for row, (_, item) in enumerate(cross_df.iterrows()):

        path1 = (
            dataset_dir
            / item["category_1"]
            / item["filename_1"]
        )

        path2 = (
            dataset_dir
            / item["category_2"]
            / item["filename_2"]
        )

        img1 = Image.open(path1).convert("L")
        img2 = Image.open(path2).convert("L")

        axes[row, 0].imshow(img1, cmap="gray")
        axes[row, 0].set_title(
            f"{item['category_1']}: "
            f"{item['filename_1']}"
        )

        axes[row, 1].imshow(img2, cmap="gray")
        axes[row, 1].set_title(
            f"{item['category_2']}: "
            f"{item['filename_2']}\n"
            f"SSIM: {item['ssim']}"
        )

        axes[row, 0].axis("off")
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()

print("\nDUPLICATE AUDIT COMPLETED.")

In [ ]:

import pandas as pd
from pathlib import Path

# ==========================================
# 1. DEFINE CONFIRMED DUPLICATE EXCLUSIONS
# ==========================================

excluded = {
    ("benign", "benign (433).png"),
    ("malignant", "malignant (145).png")
}

# ==========================================
# 2. UNION-FIND FOR DUPLICATE GROUPING
# ==========================================

parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Group high-confidence same-category candidates.
# Cross-category candidates are retained for manual review.

for _, row in audit_df.iterrows():

    a = (row["category_1"], row["filename_1"])
    b = (row["category_2"], row["filename_2"])

    if (
        row["ssim"] >= 0.90
        and row["category_1"] == row["category_2"]
    ):
        union(a, b)

# ==========================================
# 3. CREATE CLEAN DATASET MANIFEST
# ==========================================

clean_records = []

for item in records:

    key = (item["category"], item["filename"])

    if key in excluded:
        continue

    category, filename = key

    image_path = dataset_dir / category / filename

    mask_paths = sorted(
        dataset_dir.joinpath(category).glob(
            f"{Path(filename).stem}_mask*.png"
        )
    )

    group_id = str(find(key))

    clean_records.append({
        "image_id": f"{category}_{Path(filename).stem}",
        "filename": filename,
        "category": category,
        "image_path": str(image_path.resolve()),
        "mask_paths": ";".join(
            str(p.resolve()) for p in mask_paths
        ),
        "width": item["width"],
        "height": item["height"],
        "aspect_ratio": item["aspect_ratio"],
        "duplicate_group": group_id
    })

manifest = pd.DataFrame(clean_records)

# ==========================================
# 4. SAVE CROSS-CATEGORY REVIEW REPORT
# ==========================================

cross_review = audit_df[
    audit_df["cross_category"]
].copy()

# ==========================================
# 5. SAVE OUTPUT FILES
# ==========================================

output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

manifest.to_csv(
    output_dir / "dataset_manifest.csv",
    index=False
)

cross_review.to_csv(
    output_dir / "cross_category_review.csv",
    index=False
)

# ==========================================
# 6. FINAL DATASET SUMMARY
# ==========================================

print("=" * 60)
print("CLEAN DATASET MANIFEST CREATED")
print("=" * 60)

print("\nTOTAL ORIGINAL IMAGES:", len(records))
print("EXCLUDED IMAGES:", len(excluded))
print("RETAINED IMAGES:", len(manifest))

print("\nCLASS DISTRIBUTION:")
print(manifest["category"].value_counts())

print("\nUNIQUE DUPLICATE GROUPS:",
      manifest["duplicate_group"].nunique())

print("\nCROSS-CATEGORY CANDIDATES FOR REVIEW:",
      len(cross_review))

print("\nSAVED FILES:")
print(output_dir / "dataset_manifest.csv")
print(output_dir / "cross_category_review.csv")

print("\nMANIFEST PREPARATION COMPLETED.")


In [ ]:

import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from collections import defaultdict

# ============================================
# 1. CONFIGURATION
# ============================================

IMG_SIZE = 256
RANDOM_STATE = 42

processed_dir = project_dir / "data" / "processed"
output_dir = processed_dir / "preprocessed"

# ============================================
# 2. BUILD LEAKAGE-SAFE DUPLICATE GROUPS
# ============================================

parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# High-confidence same-category pairs
# All cross-category candidates are grouped
# to keep suspicious pairs in the same split.

for _, row in audit_df.iterrows():

    a = (row["category_1"], row["filename_1"])
    b = (row["category_2"], row["filename_2"])

    if row["cross_category"] or row["ssim"] >= 0.90:
        union(a, b)

# Assign group IDs
group_mapping = {}

for _, row in manifest.iterrows():

    key = (row["category"], row["filename"])
    group_mapping[key] = str(find(key))

manifest["duplicate_group"] = [
    group_mapping[(r["category"], r["filename"])]
    for _, r in manifest.iterrows()
]

# ============================================
# 3. GROUP-AWARE TRAIN / VAL / TEST SPLIT
# ============================================

splitter_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=RANDOM_STATE
)

train_val_idx, test_idx = next(
    splitter_test.split(
        manifest,
        groups=manifest["duplicate_group"]
    )
)

train_val = manifest.iloc[train_val_idx].copy()
test = manifest.iloc[test_idx].copy()

splitter_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter_val.split(
        train_val,
        groups=train_val["duplicate_group"]
    )
)

train = train_val.iloc[train_idx].copy()
val = train_val.iloc[val_idx].copy()

train["split"] = "train"
val["split"] = "validation"
test["split"] = "test"

split_manifest = pd.concat(
    [train, val, test],
    ignore_index=True
)

# Verify no duplicate group crosses splits
assert (
    split_manifest.groupby("duplicate_group")["split"].nunique().max()
    == 1
), "DATA LEAKAGE DETECTED!"

print("SPLIT DISTRIBUTION")
print(split_manifest["split"].value_counts())

print("\nCLASS DISTRIBUTION BY SPLIT")
print(pd.crosstab(
    split_manifest["split"],
    split_manifest["category"]
))

# ============================================
# 4. IMAGE AND MASK PREPROCESSING
# ============================================

def preprocess_image(image_path):

    image = Image.open(image_path).convert("L")

    w, h = image.size

    scale = min(IMG_SIZE / w, IMG_SIZE / h)

    new_w = max(1, round(w * scale))
    new_h = max(1, round(h * scale))

    image = image.resize(
        (new_w, new_h),
        Image.Resampling.BILINEAR
    )

    canvas = Image.new(
        "L",
        (IMG_SIZE, IMG_SIZE),
        0
    )

    x = (IMG_SIZE - new_w) // 2
    y = (IMG_SIZE - new_h) // 2

    canvas.paste(image, (x, y))

    return np.array(canvas)


def preprocess_mask(mask_paths, original_size):

    combined = np.zeros(
        (original_size[1], original_size[0]),
        dtype=bool
    )

    for mask_path in mask_paths:

        mask = Image.open(mask_path).convert("L")

        if mask.size != original_size:
            mask = mask.resize(
                original_size,
                Image.Resampling.NEAREST
            )

        mask_array = np.array(mask) > 0
        combined |= mask_array

    mask_image = Image.fromarray(
        (combined * 255).astype(np.uint8)
    )

    w, h = original_size

    scale = min(IMG_SIZE / w, IMG_SIZE / h)

    new_w = max(1, round(w * scale))
    new_h = max(1, round(h * scale))

    mask_image = mask_image.resize(
        (new_w, new_h),
        Image.Resampling.NEAREST
    )

    canvas = Image.new(
        "L",
        (IMG_SIZE, IMG_SIZE),
        0
    )

    x = (IMG_SIZE - new_w) // 2
    y = (IMG_SIZE - new_h) // 2

    canvas.paste(mask_image, (x, y))

    return np.array(canvas)


# ============================================
# 5. SAVE PROCESSED DATASET
# ============================================

processing_records = []

for _, row in split_manifest.iterrows():

    split = row["split"]

    image_path = Path(row["image_path"])

    mask_paths = [
        Path(p)
        for p in row["mask_paths"].split(";")
        if p
    ]

    image = preprocess_image(image_path)

    original = Image.open(image_path)
    original_size = original.size
    original.close()

    mask = preprocess_mask(
        mask_paths,
        original_size
    )

    # Verify dimensions and binary mask
    assert image.shape == (IMG_SIZE, IMG_SIZE)
    assert mask.shape == (IMG_SIZE, IMG_SIZE)
    assert set(np.unique(mask)).issubset({0, 255})

    image_dir = output_dir / split / "images"
    mask_dir = output_dir / split / "masks"

    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    image_save_path = image_dir / f"{row['image_id']}.png"
    mask_save_path = mask_dir / f"{row['image_id']}.png"

    Image.fromarray(image).save(image_save_path)
    Image.fromarray(mask).save(mask_save_path)

    processing_records.append({
        "image_id": row["image_id"],
        "category": row["category"],
        "split": split,
        "duplicate_group": row["duplicate_group"],
        "image_path": str(image_save_path),
        "mask_path": str(mask_save_path),
        "image_size": f"{IMG_SIZE}x{IMG_SIZE}",
        "mask_pixels": int(np.count_nonzero(mask))
    })


# ============================================
# 6. SAVE FINAL METADATA
# ============================================

processed_manifest = pd.DataFrame(processing_records)

split_manifest.to_csv(
    processed_dir / "dataset_splits.csv",
    index=False
)

processed_manifest.to_csv(
    processed_dir / "preprocessed_manifest.csv",
    index=False
)

# ============================================
# 7. FINAL VERIFICATION
# ============================================

print("\n" + "=" * 55)
print("PREPROCESSING COMPLETED")
print("=" * 55)

print("Original retained images:", len(manifest))
print("Processed images:", len(processed_manifest))

print("\nSplit counts:")
print(processed_manifest["split"].value_counts())

print("\nProcessed class distribution:")
print(pd.crosstab(
    processed_manifest["split"],
    processed_manifest["category"]
))

print("\nImage dimensions:")
print(processed_manifest["image_size"].value_counts())

print("\nMask foreground pixels:")
print(processed_manifest["mask_pixels"].describe())

print("\nDuplicate leakage check:")
print(
    split_manifest.groupby("duplicate_group")["split"]
    .nunique().max() == 1
)

print("\nOUTPUT DIRECTORY:")
print(output_dir)

print("\nALL FILES SAVED SUCCESSFULLY.")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import random

# ==========================================
# VISUAL QUALITY CONTROL
# ==========================================

df = processed_manifest.copy()

random.seed(42)

fig, axes = plt.subplots(4, 3, figsize=(15, 18))

classes = ["benign", "malignant", "normal"]

for row_idx, category in enumerate(classes):

    samples = df[df["category"] == category]

    if len(samples) == 0:
        continue

    selected = samples.sample(
        min(1, len(samples)),
        random_state=42
    ).iloc[0]

    image = np.array(
        Image.open(selected["image_path"]).convert("L")
    )

    mask = np.array(
        Image.open(selected["mask_path"]).convert("L")
    )

    overlay = np.stack([image] * 3, axis=-1)

    overlay[mask > 0] = (
        overlay[mask > 0] * 0.4 +
        np.array([255, 0, 0]) * 0.6
    ).astype(np.uint8)

    axes[row_idx, 0].imshow(image, cmap="gray")
    axes[row_idx, 0].set_title(f"{category.upper()} - IMAGE")

    axes[row_idx, 1].imshow(mask, cmap="gray")
    axes[row_idx, 1].set_title(f"{category.upper()} - MASK")

    axes[row_idx, 2].imshow(overlay)
    axes[row_idx, 2].set_title(f"{category.upper()} - OVERLAY")

    for ax in axes[row_idx]:
        ax.axis("off")

# Additional random sample
sample = df.sample(1, random_state=123).iloc[0]

image = np.array(
    Image.open(sample["image_path"]).convert("L")
)

mask = np.array(
    Image.open(sample["mask_path"]).convert("L")
)

axes[3, 0].imshow(image, cmap="gray")
axes[3, 0].set_title("RANDOM IMAGE")

axes[3, 1].imshow(mask, cmap="gray")
axes[3, 1].set_title("RANDOM MASK")

axes[3, 2].imshow(image, cmap="gray")
axes[3, 2].imshow(mask, cmap="jet", alpha=0.4)
axes[3, 2].set_title("RANDOM OVERLAY")

for ax in axes[3]:
    ax.axis("off")

plt.tight_layout()
plt.show()

# ==========================================
# AUTOMATED DATA VALIDATION
# ==========================================

print("\nQUALITY CONTROL REPORT")
print("=" * 50)

print("Total images:", len(df))

print("\nMissing image files:",
      sum(not Path(p).exists() for p in df["image_path"]))

print("Missing mask files:",
      sum(not Path(p).exists() for p in df["mask_path"]))

print("\nEmpty masks by class:")
print(
    df.groupby("category")["mask_pixels"]
    .apply(lambda x: (x == 0).sum())
)

print("\nNon-empty masks by class:")
print(
    df.groupby("category")["mask_pixels"]
    .apply(lambda x: (x > 0).sum())
)

print("\nMask dimensions valid:",
      all(
          Image.open(p).size == (256, 256)
          for p in df["mask_path"]
      )
)

print("\nImage dimensions valid:",
      all(
          Image.open(p).size == (256, 256)
          for p in df["image_path"]
      )
)

print("\nQC COMPLETED")
